# Road and traffic-sign segmentation: training YOLOv8n-seg

Trains a **YOLOv8 nano segmentation** model on the road dataset annotated with Roboflow (instance segmentation, 10 classes), logs the metrics to TensorBoard and exports the model to **ONNX (opset 11, 128×128)** for the TurtleBot3.

Run on Google Colab with a GPU runtime.

## 1. Environment

In [ ]:
import locale
#locale.getpreferredencoding = lambda: "UTF-8"
!pip install ultralytics==8.3.0
!pip -q show tensorboard || pip -q install tensorboard

import ultralytics
ultralytics.checks()
from ultralytics import YOLO
from torch.utils.tensorboard import SummaryWriter
import os

logdir = "/content/runs/tb_forced"
os.makedirs(logdir, exist_ok=True)
writer = SummaryWriter(logdir)

## 2. Pretrained YOLOv8n-seg model

In [ ]:
# Load a model (ici n pour nano)
# model = YOLO('yolov8n.yaml')  # build a new model from scratch
model = YOLO('yolov8n-seg.pt')  # load a pretrained model (recommended for training)

## 3. Dataset (exported from Roboflow in YOLOv8 format)

Replace the API key, workspace and project with your own.

In [ ]:
# PREPARER LE DATASET - ATTENTION, vous allez devoir adapter ces lignes de code
# en fonction des instructions obtenues lors de l'export de dataset
# en le téléchargeant depuis roboflow directement ;-)
from google.colab import drive
drive.mount('/content/drive')

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("maquette_traffic")
version = project.version(5)
dataset = version.download("yolov8")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Training (100 epochs, 128×128, `overlap_mask=True`)

In [ ]:
# CHANGEZ LE CHEMIN D'ACCESS AU DATASET en fonction de son emplacement sur votre espace
# ATTENTION - il est possible que vous allez devoir changer les chemins d'accès vers les répertoires train et validation
def on_fit_epoch_end(trainer):
    epoch = trainer.epoch
    metrics = trainer.metrics  # dict
    for k, v in metrics.items():
        if isinstance(v, (int, float)):
            writer.add_scalar(k, v, epoch)

model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

model.train(
    overlap_mask=True,
    data="/content/Maquette_Traffic-5/data.yaml",
    epochs=100,
    imgsz=128,
)

writer.close()

In [ ]:

%load_ext tensorboard
%tensorboard --logdir /content/runs/

## 5. Export for the robot

The ONNX export is done with **opset 11**, the version supported by the runtime on the robot.

In [ ]:
#EXPORTER le modèle pour l'utilisation :
# Install onnxscript explicitly
!pip install onnxscript

model.export()
model.export(format="onnx", opset=11, simplify=True, device='cpu',imgsz=[128,128])
#le modèle est sauvegardé dans runs/detect/train.....   avec extension .pt si pytorch choissi.
#modifier le chemin pour sauvegarder le bon fichier - qui correspond à votre meuilleur modèle
!cp '/content/runs/segment/train3/weights/best.pt' '/content/drive/MyDrive/road_tracking'
!cp '/content/runs/segment/train3/weights/best.onnx' '/content/drive/MyDrive/road_tracking'

# First, make sure you have your original trained PyTorch model file.
# For example, if your PyTorch model is named 'best.pt'

# Load the PyTorch model
model_pt = YOLO('/content/drive/MyDrive/road_tracking/best.pt') # Replace with the actual path to your .pt model

# Export the model to ONNX with opset=11
# You might need to adjust the name if you want to keep the old one
model_pt.export(format='onnx', opset=11, simplify=True, name='best_opset11.onnx')
from google.colab import files
files.download("runs/segment/train/results.png")
